# LangChain LCEL

### 1. LCEL 의 기본 구조를 배워 봅시다.
1. 프롬프트 템플릿 정하기
2. 모델 정하기
3. 출력 형태 정하기

In [1]:
# LangSmith 추적 설정 부분
from dotenv import load_dotenv
import os

load_dotenv()

project_name = "wanted_2nd_langchain_basic"
os.environ["LANGSMITH_PROJECT"] = project_name

### 1. 프롬프트 템플릿 설정하기

In [7]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
template = "최근 {year}년에 동안 {country}에 오는 관광객 수는 얼마인지 알려주세요"
prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['country', 'year'], input_types={}, partial_variables={}, template='최근 {year}년에 동안 {country}에 오는 관광객 수는 얼마인지 알려주세요')

In [8]:
prompt_template.format(year=5, country="뉴욕")

'최근 5년에 동안 뉴욕에 오는 관광객 수는 얼마인지 알려주세요'

### 2. 모델 선택하기

In [9]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [10]:
# 모델과 프롬프트 템플릿 연결
chain = prompt_template | model
chain

PromptTemplate(input_variables=['country', 'year'], input_types={}, partial_variables={}, template='최근 {year}년에 동안 {country}에 오는 관광객 수는 얼마인지 알려주세요')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001A3FB583E90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001A3FB596D50>, root_client=<openai.OpenAI object at 0x000001A3FF814610>, root_async_client=<openai.AsyncOpenAI object at 0x000001A3FA8EECD0>, model_name='gpt-4.1-mini', temperature=0.1, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [12]:
input = {"year": 3, "country": "뉴욕"}
answer = chain.invoke(input)
print(answer.content)

최근 3년간 뉴욕을 방문한 관광객 수에 대한 최신 통계는 다음과 같습니다:

- 2021년: 약 3,800만 명  
- 2022년: 약 5,000만 명  
- 2023년: 약 6,000만 명 (예상치 및 초기 집계 기준)

코로나19 팬데믹의 영향으로 2020년과 2021년에는 관광객 수가 크게 감소했으나, 2022년부터 점차 회복세를 보이고 있습니다. 2023년에는 팬데믹 이전 수준에 근접하거나 이를 넘어서는 방문객 수가 예상됩니다.

정확한 최신 수치는 뉴욕시 관광청(NYC & Company) 공식 보고서를 참고하시면 좋습니다.


### 3. 출력 양식 정해보기
- Stroutputparser
- jsonoutputparser
- pydanticoutputparser
- commaseperateoutputparser

In [15]:
from langchain_core.output_parsers import StrOutputParser

outputparser = StrOutputParser()
# prompt template + model + outputparser 세개를 연결한 chain - 기본적인 lcel 구조
chain = prompt_template | model | outputparser

input = {"year": 2, "country": "방콕"}
answer = chain.invoke(input)
# outputparser로 content만 빼왔기때문에 .content 안해도됨
print(answer)

최근 2년 동안 방콕을 방문한 관광객 수에 대한 최신 통계는 2022년과 2023년 데이터를 기준으로 말씀드릴 수 있습니다.

- 2022년: 코로나19 팬데믹 이후 국제 여행이 점차 회복되면서 방콕을 방문한 관광객 수는 약 6백만 명 정도로 추정됩니다.
- 2023년: 여행 수요가 더욱 증가하여 약 1,000만 명 이상의 관광객이 방콕을 방문한 것으로 예상됩니다.

정확한 수치는 태국 관광청(TAT)이나 방콕시 관광 관련 공식 통계 자료를 참고하시는 것이 좋습니다. 팬데믹 이전인 2019년에는 방콕을 방문한 관광객 수가 약 2,200만 명에 달했으나, 이후 코로나19 영향으로 크게 감소했다가 점차 회복 중입니다.


### 실습
- 각자 알아서 질문 주제 정하고
- 프롬프트 템플릿을 좀 더 자세하게 작성해보기

In [ ]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
template = """
최근 {time_range} 동안 {category} 분야에서 가장 주목받는 기업 {n}개를 선정해주세요.  
선정 기준은 투자 유치 규모, 기술 혁신성, 시장 점유율 성장, 미디어 언급 등을 종합적으로 고려해주세요.  

응답 형식은 다음과 같이 표 형태로 작성해주세요:
1. 기업명
2. 국가/지역
3. 주요 제품/서비스
4. 최근 성과 및 특징

"""
prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['category', 'n', 'time_range'], input_types={}, partial_variables={}, template='\n최근 {time_range} 동안 {category} 분야에서 가장 주목받는 기업 {n}개를 선정해주세요.  \n선정 기준은 투자 유치 규모, 기술 혁신성, 시장 점유율 성장, 미디어 언급 등을 종합적으로 고려해주세요.  \n\n응답 형식은 다음과 같이 표 형태로 작성해주세요:\n1. 기업명\n2. 국가/지역\n3. 주요 제품/서비스\n4. 최근 성과 및 특징\n\n')

In [20]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    temperature=0.1,
    model="gpt-4.1-mini",
    verbose=True
)

In [21]:
from langchain_core.output_parsers import StrOutputParser

outputparser = StrOutputParser()
# prompt template + model + outputparser 세개를 연결한 chain - 기본적인 lcel 구조
chain = prompt_template | model | outputparser

In [23]:
input = {"time_range":1, "category": "AI", "n": 10}
answer = chain.invoke(input)
# outputparser로 content만 빼왔기때문에 .content 안해도됨
print(answer)

아래는 최근 1년(2023년 중반~2024년 중반) 동안 AI 분야에서 가장 주목받는 10개 기업을 투자 유치 규모, 기술 혁신성, 시장 점유율 성장, 미디어 언급 등을 종합적으로 고려하여 선정한 목록입니다.

| 순위 | 기업명           | 국가/지역       | 주요 제품/서비스                         | 최근 성과 및 특징                                                                                   |
|-------|------------------|-----------------|----------------------------------------|--------------------------------------------------------------------------------------------------|
| 1     | OpenAI           | 미국            | GPT 시리즈, ChatGPT, DALL·E 등         | GPT-4 출시 및 API 확장, 마이크로소프트와의 전략적 파트너십 강화, 대규모 투자 유치 및 시장 영향력 확대          |
| 2     | NVIDIA           | 미국            | AI 칩셋, GPU, AI 컴퓨팅 플랫폼          | AI 하드웨어 수요 급증, AI 전용 GPU(A100, H100) 판매 호조, AI 인프라 시장 점유율 선도                        |
| 3     | Anthropic        | 미국            | AI 안전성 및 윤리적 AI 모델 개발         | 3억 달러 이상 투자 유치, AI 안전성에 중점 둔 대형 언어 모델 개발, 시장 내 신뢰성 확보 노력                     |
| 4     | Google DeepMind  | 영국/미국       | AI 연구,